# 02 Feature engineering and split

Day-ahead setup: each day is forecast using data up to 23:00 of the previous day. This notebook builds the feature table for the Duomo zone, fixes the train / test / stress-test windows and checks that no feature uses information from the forecast horizon.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from features import make_features, check_no_leakage
from config import TRAIN_END, TEST_START, TEST_END, STRESS_START

zones = pd.read_csv("../data/processed/milan_zones_hourly.csv", index_col="ts", parse_dates=True)

## Features

All features are available a day ahead, so the smallest lag is 24 h and rolling windows are shifted by 24 h.

- `lag_24`, `lag_48`, `lag_168` : traffic 1 day, 2 days and 1 week earlier 
- `roll_mean_24`, `roll_mean_168` : mean of the 24 / 168 hours ending 24 h before the target hour 
- `hour_sin`, `hour_cos` : cyclical encoding of the hour of day
- `day_type` : day of week (0 = Mon … 6 = Sun); holidays are mapped to 6 
- `is_weekend` : 1 if `day_type` is Saturday or Sunday 
- `is_holiday` : 1 on public holidays (1 Nov, 7–8 Dec, 25–26 Dec, 1 Jan) 
- `roll_mean_168` : needs 168 hours of history that ends 24 hours before the target, so the first 191 hours are dropped (1,488 → 1,297 rows).

In [2]:
feat = make_features(zones["duomo"])
print(feat.shape)
feat.head()

(1297, 11)


,y,lag_24,lag_48,lag_168,roll_mean_24,roll_mean_168,hour_sin,hour_cos,day_type,is_weekend,is_holiday
ts,,,,,,,,,,,
2013-11-08 23:00:00,25390.321436,29822.941425,26400.052958,19962.459163,57488.602478,48792.216142,-0.258819,0.965926,4,0,0
2013-11-09 00:00:00,21092.321899,23103.824557,19268.416949,15358.918787,57648.411128,48793.821129,0.000000,1.000000,5,1,0
2013-11-09 01:00:00,20437.479092,18366.507316,15696.002804,13584.569541,57759.682149,48766.152827,0.258819,0.965926,5,1,0
2013-11-09 02:00:00,23525.625241,13567.498157,15983.303282,10490.904799,57659.023602,48717.779038,0.500000,0.866025,5,1,0
2013-11-09 03:00:00,21539.935019,13569.500063,13358.058018,9631.755898,57667.833688,48684.565247,0.707107,0.707107,5,1,0


## Train / test split

- Train: 8 Nov 23:00 – 8 Dec 23:00.
- Test: 9–19 Dec, an ordinary period without holidays.
- Holiday stress test: 20 Dec – 1 Jan. This period was inspected during exploration, so it is reported separately and not used for model selection.

The split is chronological; nothing is shuffled.

In [3]:
splits = {
    "train": feat.loc[:TRAIN_END],
    "test": feat.loc[TEST_START:TEST_END],
    "holiday stress": feat.loc[STRESS_START:],
}
pd.DataFrame({name: {"start": df.index.min(), "end": df.index.max(), "hours": len(df)}
              for name, df in splits.items()}).T

,start,end,hours
train,2013-11-08 23:00:00,2013-12-08 23:00:00,721
test,2013-12-09 00:00:00,2013-12-19 23:00:00,264
holiday stress,2013-12-20 00:00:00,2014-01-01 23:00:00,312


## Holiday coverage in the training window

All training-period holidays (7–8 Dec) fall on a weekend and the test window contains none, so weekday-holiday behaviour cannot be learned from the data. Weekday holidays are therefore mapped to the Sunday schedule (`holiday_as_sunday=True`) and assessed separately in the holiday stress test.

In [4]:
tr = feat.loc[:TRAIN_END]
hol_dow = tr.index[tr["is_holiday"] == 1].day_of_week
print("Training holiday hours by weekday (5=Sat, 6=Sun):")
print(pd.Series(hol_dow).value_counts().sort_index())
print("Holiday hours in test window:", feat.loc[TEST_START:TEST_END, "is_holiday"].sum())

Training holiday hours by weekday (5=Sat, 6=Sun):
ts
5    24
6    24
Name: count, dtype: int64
Holiday hours in test window: 0


## Leakage check

For a chosen timestamp t, every observation from t − 23 h onwards is corrupted and the features are rebuilt. The features at t must stay unchanged; otherwise a feature uses data from inside the forecast horizon. The check is repeated at three timestamps.

In [5]:
for pos in (200, 600, 1000):
    check_no_leakage(zones["duomo"], pos=pos)

no leakage detected at 2013-11-17 07:00:00
no leakage detected at 2013-12-03 23:00:00
no leakage detected at 2013-12-20 15:00:00
